<a href="https://colab.research.google.com/github/Anirudh-R-1201/Football_xG/blob/master/statsbomb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab
%matplotlib inline
import seaborn as sns
!pip install scikit-plot
import scikitplot as skplt

from sklearn import preprocessing, model_selection, metrics
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

In [ ]:
!pip install statsbomb
import statsbomb as sb

In [ ]:
comps = sb.Competitions()
#json_data = comps.data
df = comps.get_dataframe()
df

In [ ]:
ucl_matches_1718 = sb.Matches(event_id='16', season_id='4').get_dataframe()
ucl_matches_1819 = sb.Matches(event_id='16', season_id='1').get_dataframe()
wc_matches = sb.Matches(event_id='43', season_id='3').get_dataframe()

matches = ucl_matches_1718.append(ucl_matches_1819).append(wc_matches)
matches.head()

In [ ]:
match_list = matches['match_id'].tolist()

shots_df = pd.DataFrame()
for i in match_list:
    events = sb.Events(event_id='%s' % i)
    df = events.get_dataframe(event_type='shot')
    shots_df = shots_df.append(df)
    
shots_df.head()

In [ ]:
shots_df.shape

In [ ]:
from google.colab import files
shots_df.to_csv('shots_df.csv') 
files.download('shots_df.csv')

In [ ]:
print(len(shots_df), "shots ")
print("")

print("Columns :")
print(list(shots_df))
print("")

print("Unique Types :")
print(shots_df['type'].unique())
print("")

print("Unique Play Patterns :")
print(shots_df['play_pattern'].unique())

In [ ]:
np_shots = shots_df[shots_df['type'] != 'Penalty']
np_shots['goal'] = np.where(np_shots['outcome'] == 'Goal', 1, 0)

attempts = len(np_shots)
goals = sum(np_shots['goal'])
misses = attempts - goals
conversion_rate = goals / attempts
print("Conversion rate:", "{0:.2f}%".format(conversion_rate*100))
print("")

sns.set(style="ticks", color_codes=True)
sns.countplot(x="goal", data=np_shots)

In [ ]:
np_shots = np_shots.reset_index().drop('level_0', axis=1)

np_shots['assisted'] = np.where(np_shots['key_pass_id'].isna(), 0, 1)

np_shots['x_distance'] = 120 - np_shots['start_location_x']
np_shots['y_distance'] = abs(40 - np_shots['start_location_y'])
np_shots['distance'] = np.sqrt((np_shots['x_distance']**2 + np_shots['y_distance']**2))
np_shots['angle'] = np.degrees(np.arctan((np_shots['y_distance'] / np_shots['x_distance'])))

np_shots['body_part'] = np.where((np_shots['body_part'] == 'Right Foot')| (np_shots['body_part'] == 'Left Foot'), 'foot',
                                 np.where(np_shots['body_part'] == 'Head', 'head', 'other'))

In [ ]:
np_shots.head()

In [ ]:
np_shots = np_shots.fillna(0)

In [ ]:
from google.colab import files
np_shots.to_csv('np_shots.csv') 
files.download('np_shots.csv')

In [ ]:
feature_cols = ['play_pattern', 'under_pressure', 'body_part', 'technique', 'first_time',
                'follows_dribble', 'redirect', 'one_on_one', 'open_goal', 'deflected',
                'assisted', 'distance', 'angle']

features = np_shots[feature_cols]
labels = np_shots['goal']

features = features.fillna(0)
labels = labels.fillna(0)

In [ ]:
cat_cols = ['play_pattern', 'under_pressure', 'body_part', 'technique', 'first_time',
                'follows_dribble', 'redirect', 'one_on_one', 'open_goal', 'deflected']

cat_features = features[cat_cols]
features = features.drop(cat_cols, axis=1)

le = preprocessing.LabelEncoder()
cat_features = cat_features.apply(le.fit_transform)
features = features.merge(cat_features, left_index=True, right_index=True)

In [ ]:
features = features.sample(frac=1)
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.25, shuffle=True, random_state=42)

scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)
print(X_train.shape)
print(X_test.shape)

In [ ]:
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_pred_prob = clf.predict_proba(X_test)

In [ ]:
print("Predicted XG of test set:", sum(y_pred))
print("xG :", "{0:.2f}".format(sum(y_pred_prob[:,1])))
print("Actual goals test set:", sum(y_test))
print('')
print(metrics.classification_report(y_test, y_pred))

skplt.metrics.plot_confusion_matrix(y_test, y_pred)
skplt.metrics.plot_precision_recall(y_test, y_pred_prob)

importances = pd.DataFrame({'feature':features.columns,'importance':np.round(clf.feature_importances_,3)})
importances = importances.sort_values('importance',ascending=False)

f, ax = plt.subplots(figsize=(6, 8))
g = sns.barplot(x='importance', y='feature', data=importances, color="blue", saturation=.2, label="Total")
g.set(xlabel='Feature Importance', ylabel='Feature', title='Feature Importance in Predicting xG')
plt.show()

In [ ]:
from imblearn.over_sampling import SMOTENC
import warnings
warnings.filterwarnings("ignore")

In [ ]:
cat_cols_ind = []
for key in cat_cols:
    ind = features.columns.get_loc(key)
    cat_cols_ind.append(ind)

smote_nc = SMOTENC(categorical_features=cat_cols_ind, random_state=42)
features_resampled, labels_resampled = smote_nc.fit_resample(features, labels)

X_train_re, X_test_re, y_train_re, y_test_re = train_test_split(features_resampled, labels_resampled,test_size=0.25, shuffle=True, random_state=42)

scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
X_train_re = scaler.fit_transform(X_train_re)
X_test_re = scaler.fit_transform(X_test_re)
print(X_train_re.shape)
print(X_test_re.shape)

In [ ]:
clf = DecisionTreeClassifier(random_state=42)

clf.fit(X_train_re, y_train_re)

y_pred_re = clf.predict(X_test_re)
y_pred_prob_re = clf.predict_proba(X_test_re)

print("Predicted goals on test set:", sum(y_pred_re))
print("xG :", "{0:.2f}".format(sum(y_pred_prob_re[:,1])))
print("Actual goals :", sum(y_test_re))
print('')
print(metrics.classification_report(y_test_re, y_pred_re))

skplt.metrics.plot_confusion_matrix(y_test_re, y_pred_re)

skplt.metrics.plot_precision_recall(y_test_re, y_pred_prob_re)

importances = pd.DataFrame({'feature':features.columns,'importance':np.round(clf.feature_importances_,3)})
importances = importances.sort_values('importance',ascending=False)

f, ax = plt.subplots(figsize=(6, 8))
g = sns.barplot(x='importance', y='feature', data=importances,color="blue", saturation=.2, label="Total")
g.set(xlabel='Feature Importance', ylabel='Feature', title='Feature Importance in Predicting xG')
plt.show()